# Deduplicate Aux File
## Remove duplicate vin_date rows

In [ ]:
import pandas as pd
import yaml
import time
import os

if os.path.basename(os.getcwd()) == 'debug':
    os.chdir('..')
print(f'Working directory: {os.getcwd()}')

In [ ]:
with open('config/car_liab/v1/config.yaml') as f:
    cfg = yaml.safe_load(f)
with open('current.pc') as f:
    pc = f'PC{f.read().strip()}'

aux_path = cfg['machines'][pc]['paths']['aux_data_path']

# Use ORIGINAL file (not the one in config)
file_path = f'{aux_path}/car_with_dep_factor_fold.parquet'
output_path = f'{aux_path}/car_with_dep_factor_fold_dedup.parquet'

print(f'Source: {file_path}')
print(f'Output: {output_path}')

In [ ]:
print('Loading...')
start = time.time()
df = pd.read_parquet(file_path)
print(f'Loaded in {time.time()-start:.1f}s')
print(f'Shape: {df.shape[0]:,} × {df.shape[1]}')
print(f'Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')

total = len(df)
unique = df['vin_date'].nunique()
dups = total - unique
print(f'Total: {total:,}')
print(f'Unique: {unique:,}')
print(f'Duplicates: {dups:,} ({100*dups/total:.1f}%)')

In [ ]:
print('='*70)
print('VERIFICATION: Hash-based check')
print('='*70)
start = time.time()

cols_to_hash = [c for c in df.columns if c != 'vin_date']
row_hashes = pd.util.hash_pandas_object(df[cols_to_hash], index=False)
hash_df = pd.DataFrame({'vin_date': df['vin_date'], 'hash': row_hashes})
hash_nunique = hash_df.groupby('vin_date')['hash'].nunique()
mismatches = hash_nunique[hash_nunique > 1]

print(f'Time: {time.time()-start:.1f}s')
print(f'Mismatched groups: {len(mismatches):,}')
if len(mismatches) == 0:
    print('✓ ALL DUPLICATES IDENTICAL - Safe to drop')
else:
    print('✗ WARNING: Some rows differ!')
    print(mismatches.head(10))

del row_hashes, hash_df, hash_nunique
import gc; gc.collect()

In [ ]:
print('='*70)
print('DEDUPLICATION')
print('='*70)
start = time.time()

df_dedup = df.drop_duplicates(subset=['vin_date'], keep='first')
print(f'Time: {time.time()-start:.1f}s')
print(f'Before: {len(df):,} rows')
print(f'After: {len(df_dedup):,} rows')
print(f'Removed: {len(df)-len(df_dedup):,}')

print(f'Saving to: {output_path}')
start = time.time()
df_dedup.to_parquet(output_path, index=False, compression='snappy')
print(f'✓ Saved in {time.time()-start:.1f}s')

orig_size = os.path.getsize(file_path) / 1e9
new_size = os.path.getsize(output_path) / 1e9
print(f'Original: {orig_size:.2f} GB')
print(f'Dedup: {new_size:.2f} GB ({100*new_size/orig_size:.1f}%)')

In [ ]:
print('='*70)
print('VERIFICATION: Checking saved file')
print('='*70)
df_check = pd.read_parquet(output_path)
print(f'Shape: {df_check.shape}')
print(f'Unique vin_dates: {df_check["vin_date"].nunique():,}')
dups_check = df_check.duplicated(subset=['vin_date']).sum()
print(f'Duplicates: {dups_check:,}')

if dups_check == 0:
    print('\n✓✓✓ SUCCESS - File is clean!')
else:
    print('\n✗ WARNING: Still has duplicates!')

print('\nNext: Update config files (line 71):')
print('  config/car_liab/v1/config.yaml')
print('  config/car_coll/v1/config.yaml')
print('Change:')
print('  FROM: aux_file: "car_with_dep_factor_fold.parquet"')
print('  TO:   aux_file: "car_with_dep_factor_fold_dedup.parquet"')